AULA 6

PRÉ PROCESSAMENTO

In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# arquivo de concessão de crédito simplificado, com algumas colunas a menos
dataset = pd.read_csv('credit_simple.csv', sep=';')

#1000 linhas e 8 atribus
dataset.shape

# analisar primeiros itens
dataset.head()

# separar a classe das variáveis independentes
y = dataset['CLASSE']
X = dataset.iloc[:,:-1]

In [3]:
# Analisar primeiros itens
dataset.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,CLASSE
0,1169.0,4,67,nenhum,01/01/2019,masculino solteiro,radio/tv,bom
1,5951.0,2,22,nenhum,01/01/2020,fem div/cas,radio/tv,ruim
2,2096.0,3,49,nenhum,02/01/2020,masculino solteiro,educação,bom
3,7882.0,4,45,nenhum,02/01/2019,masculino solteiro,mobilia/equipamento,bom
4,4870.0,4,53,nenhum,03/01/2018,masculino solteiro,carro novo,ruim


In [7]:
# separar a classe das variáveis independentes
y = dataset['CLASSE']
X = dataset.iloc[:,:-1]

In [6]:
# verificar valores faltantes -> nulos!
# saldo atual (numerico) e estado civil (categorico)
X.isnull().sum()
# saldo atual -> pode usar mediana
# melhor que média porque não esta suceptivel a valores extremos
# como um bilionario
# como trata-se do modelo, um treinamento, isso pode ser feito aqui
# não pode ser feito no BD, por exemplo

,0
SALDO_ATUAL,7
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,8
PROPOSITO,0


In [9]:
mediana = X['SALDO_ATUAL'].median()
mediana

2323.0

In [12]:
# fillna -> preencha valores NA e atualize os arquivos originais!
X['SALDO_ATUAL'].fillna(mediana, inplace=True)
# validar novamente para verificar se existem valores em branco
X.isnull().sum()

/tmp/ipykernel_3789/2491375268.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['SALDO_ATUAL'].fillna(mediana, inplace=True)


,0
SALDO_ATUAL,0
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,8
PROPOSITO,0


In [13]:
# tecnica para valores categoricos não preenchidos -> usar a moda!
# com "group by" da para encontrar os valores que mais aparecem
agrupado = X.groupby(['ESTADOCIVIL']).size()
agrupado

,0
ESTADOCIVIL,
fem div/cas,308
masculino casado/viuvo,92
masculino div/sep,50
masculino solteiro,542


In [14]:
X['ESTADOCIVIL'].fillna('masculino solteiro', inplace=True)
# verificar logo na sequência
X.isnull().sum()

/tmp/ipykernel_3789/3384672102.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['ESTADOCIVIL'].fillna('masculino solteiro', inplace=True)


,0
SALDO_ATUAL,0
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,0
PROPOSITO,0


In [16]:
# ex.: Criar regressão linear prevendo saldo atual -> um outlier alto influencia!
# aqui usaremos o critério de, se o valor for maior ou igual a 2 desvios padrões
# então será considerado um outlier
desv = X['SALDO_ATUAL'].std()
desv

2808.9673591141222

In [17]:
# índice 127 e índice 160
X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL']

,SALDO_ATUAL
1,5951.0
3,7882.0
5,9055.0
7,6948.0
17,8072.0
...,...
973,7297.0
980,8386.0
983,8229.0
986,6289.0


In [18]:
# atualizar esses valores para a mediana!
mediana = X['SALDO_ATUAL'].median()
mediana

2323.0

In [21]:
# fazendo a substituição pela mediana
X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL'] = mediana
X.loc[X['SALDO_ATUAL']>= 2 * desv]

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO


In [22]:
# data binning:
# vamos analisar itens que foram digitados errados
# itens que tem pouca relevância
# pegar essas categorias e atribuir a "outros"
agrupado = X.groupby(['PROPOSITO']).size()
agrupado

,0
PROPOSITO,
Eletrodomésticos,12
carro novo,234
carro usado,103
educação,50
mobilia/equipamento,181
negócios,97
obras,22
outros,12
qualificação,9


In [23]:
# podemos transformar eletrodomésticos e qualificação em "outros"
X.loc[X['PROPOSITO']=='Eletrodomésticos','PROPOSITO'] = 'outros'
X.loc[X['PROPOSITO']=='qualificação','PROPOSITO'] = 'outros'
agrupado = X.groupby(['PROPOSITO']).size()
agrupado

,0
PROPOSITO,
carro novo,234
carro usado,103
educação,50
mobilia/equipamento,181
negócios,97
obras,22
outros,33
radio/tv,280


In [25]:
# extração de características - exemplo da data!
# pode separar dia, mês e ano
# se é feriado, se faz parte do fds
# em que trimestre fiscal está no ano, etc.
X['DATA']

,DATA
0,2019-01-01
1,2020-01-01
2,2020-01-02
3,2019-01-02
4,2018-01-03
...,...
995,2018-06-29
996,2018-06-30
997,2018-07-03
998,2019-07-04


In [26]:
# transformada a data, vamos armazenar ela na mesma coluna
X['DATA'] = pd.to_datetime(X['DATA'], format='%d/%m/%Y')
X['DATA']

,DATA
0,2019-01-01
1,2020-01-01
2,2020-01-02
3,2019-01-02
4,2018-01-03
...,...
995,2018-06-29
996,2018-06-30
997,2018-07-03
998,2019-07-04


In [28]:
X['ANO'] = X['DATA'].dt.year
X['MES'] = X['DATA'].dt.month
X['DIASEMANA'] = X['DATA'].dt.day_name()
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,masculino solteiro,radio/tv,2019,1,Tuesday
1,2323.0,2,22,nenhum,2020-01-01,fem div/cas,radio/tv,2020,1,Wednesday
2,2096.0,3,49,nenhum,2020-01-02,masculino solteiro,educação,2020,1,Thursday
3,2323.0,4,45,nenhum,2019-01-02,masculino solteiro,mobilia/equipamento,2019,1,Wednesday
4,4870.0,4,53,nenhum,2018-01-03,masculino solteiro,carro novo,2018,1,Wednesday
...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,fem div/cas,mobilia/equipamento,2018,6,Friday
996,3857.0,4,40,nenhum,2018-06-30,masculino div/sep,carro usado,2018,6,Saturday
997,804.0,4,38,nenhum,2018-07-03,masculino solteiro,radio/tv,2018,7,Tuesday
998,1845.0,4,23,nenhum,2019-07-04,masculino solteiro,radio/tv,2019,7,Thursday


In [29]:
# labelencoder - atribuir para informações categóricas um valor numérico
# alguns modelos convertem internamente, porém outros não

X['ESTADOCIVIL'].unique() # semelhante ao distinct do SELECT

array(['masculino solteiro', 'fem div/cas', 'masculino div/sep',
       'masculino casado/viuvo'], dtype=object)

In [30]:
# one hot encoding transformaria atributos em colunas -> não tão legal aqui
# usar no futuro para outros planos de pagamento
X['PROPOSITO'].unique()

X['DIASEMANA'].unique()

array(['Tuesday', 'Wednesday', 'Thursday', 'Saturday', 'Sunday', 'Monday',
       'Friday'], dtype=object)

In [32]:
labelencoder1 = LabelEncoder()
X['ESTADOCIVIL'] = labelencoder1.fit_transform(X['ESTADOCIVIL'])
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5
1,2323.0,2,22,nenhum,2020-01-01,0,7,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4
3,2323.0,4,45,nenhum,2019-01-02,3,3,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6
...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,0,3,2018,6,0
996,3857.0,4,40,nenhum,2018-06-30,2,1,2018,6,2
997,804.0,4,38,nenhum,2018-07-03,3,7,2018,7,5
998,1845.0,4,23,nenhum,2019-07-04,3,7,2019,7,4


In [33]:
X['PROPOSITO'] = labelencoder1.fit_transform(X['PROPOSITO'])
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5
1,2323.0,2,22,nenhum,2020-01-01,0,7,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4
3,2323.0,4,45,nenhum,2019-01-02,3,3,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6
...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,0,3,2018,6,0
996,3857.0,4,40,nenhum,2018-06-30,2,1,2018,6,2
997,804.0,4,38,nenhum,2018-07-03,3,7,2018,7,5
998,1845.0,4,23,nenhum,2019-07-04,3,7,2019,7,4


In [34]:
X['DIASEMANA'] = labelencoder1.fit_transform(X['DIASEMANA'])
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5
1,2323.0,2,22,nenhum,2020-01-01,0,7,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4
3,2323.0,4,45,nenhum,2019-01-02,3,3,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6
...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,0,3,2018,6,0
996,3857.0,4,40,nenhum,2018-06-30,2,1,2018,6,2
997,804.0,4,38,nenhum,2018-07-03,3,7,2018,7,5
998,1845.0,4,23,nenhum,2019-07-04,3,7,2019,7,4


In [35]:
# one hot encoding
# analisar a cardinalidade
X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5
1,2323.0,2,22,nenhum,2020-01-01,0,7,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4
3,2323.0,4,45,nenhum,2019-01-02,3,3,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6


In [36]:
# existem apenas 3 valores:
# 'nenhum', 'banco', 'stores'
outros = X['OUTROSPLANOSPGTO'].unique()
outros

array(['nenhum', 'banco', 'stores'], dtype=object)

In [37]:
# ao aplicar ohe definimos o prefixo para a coluna
z = pd.get_dummies(X['OUTROSPLANOSPGTO'], prefix = 'OUTROS')
z

,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False
...,...,...,...
995,False,True,False
996,False,True,False
997,False,True,False
998,False,True,False


In [38]:
# fazer a análise do objeto X -> existem atributos que precisam estar padronizados
# saldo atual, residenciadesde, idade
# e juntaremos com o Z criado anteriormente!
sc = StandardScaler()
m = sc.fit_transform(X.iloc[:,0:3])
# m é um objeto numpy, não um dataframe
# ele não tem o nome das colunas
m

array([[-0.97243744,  1.04698668,  1.6392759 ],
       [ 0.07120363, -0.76597727, -0.74024139],
       [-0.13408798,  0.14050471,  0.68746898],
       ...,
       [-1.30253188,  1.04698668,  0.1058092 ],
       [-0.36108444,  1.04698668, -0.68736323],
       [ 2.10874551,  1.04698668, -0.47585058]])

In [39]:
# para contatenar, precisa transformar o m em dataframe
X = pd.concat([X,z,pd.DataFrame(m,columns=['SALDO_ATUAL_N','RESIDENCEADESDE_N','IDADE_N'])],axis=1)

X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA,OUTROS_banco,OUTROS_nenhum,OUTROS_stores,SALDO_ATUAL_N,RESIDENCEADESDE_N,IDADE_N
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5,False,True,False,-0.972437,1.046987,1.639276
1,2323.0,2,22,nenhum,2020-01-01,0,7,2020,1,6,False,True,False,0.071204,-0.765977,-0.740241
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4,False,True,False,-0.134088,0.140505,0.687469
3,2323.0,4,45,nenhum,2019-01-02,3,3,2019,1,6,False,True,False,0.071204,1.046987,0.475956
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6,False,True,False,2.374630,1.046987,0.898982


In [40]:
# tem várias colunas em X repetidas, vamos excluir elas do dataframe
# colunas que fizemos ohe, data, etc.
# apenas do ohe, ainda tem uma coluna que pode ser removida "por tabela"
X.drop(columns=['SALDO_ATUAL','RESIDENCIADESDE','IDADE','OUTROSPLANOSPGTO','DATA','OUTROS_banco'], inplace=True)
X

,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA,OUTROS_nenhum,OUTROS_stores,SALDO_ATUAL_N,RESIDENCEADESDE_N,IDADE_N
0,3,7,2019,1,5,True,False,-0.972437,1.046987,1.639276
1,0,7,2020,1,6,True,False,0.071204,-0.765977,-0.740241
2,3,2,2020,1,4,True,False,-0.134088,0.140505,0.687469
3,3,3,2019,1,6,True,False,0.071204,1.046987,0.475956
4,3,0,2018,1,6,True,False,2.374630,1.046987,0.898982
...,...,...,...,...,...,...,...,...,...,...
995,0,3,2018,6,0,True,False,-0.459661,1.046987,-0.264338
996,2,1,2018,6,2,True,False,1.458505,1.046987,0.211566
997,3,7,2018,7,5,True,False,-1.302532,1.046987,0.105809
998,3,7,2019,7,4,True,False,-0.361084,1.046987,-0.687363


PCA - Principal Component Analysis

In [41]:
from sklearn.model_selection import train_test_split # dividir os dados entre treino e teste
from sklearn.metrics import accuracy_score # métrica
from sklearn.decomposition import PCA # técnica aqui utilizada
from sklearn.ensemble import RandomForestClassifier # modelo a ser utilizado
from sklearn.preprocessing import StandardScaler # padronização de dados
from sklearn import datasets

iris = datasets.load_iris()
previsores = iris.data
classe = iris.target
previsores

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [42]:
# deixando os dados padronizados
sc = StandardScaler()
previsores = sc.fit_transform(previsores)
previsores

array([[-9.00681170e-01,  1.01900435e+00, -1.34022653e+00,
        -1.31544430e+00],
       [-1.14301691e+00, -1.31979479e-01, -1.34022653e+00,
        -1.31544430e+00],
       [-1.38535265e+00,  3.28414053e-01, -1.39706395e+00,
        -1.31544430e+00],
       [-1.50652052e+00,  9.82172869e-02, -1.28338910e+00,
        -1.31544430e+00],
       [-1.02184904e+00,  1.24920112e+00, -1.34022653e+00,
        -1.31544430e+00],
       [-5.37177559e-01,  1.93979142e+00, -1.16971425e+00,
        -1.05217993e+00],
       [-1.50652052e+00,  7.88807586e-01, -1.34022653e+00,
        -1.18381211e+00],
       [-1.02184904e+00,  7.88807586e-01, -1.28338910e+00,
        -1.31544430e+00],
       [-1.74885626e+00, -3.62176246e-01, -1.34022653e+00,
        -1.31544430e+00],
       [-1.14301691e+00,  9.82172869e-02, -1.28338910e+00,
        -1.44707648e+00],
       [-5.37177559e-01,  1.47939788e+00, -1.28338910e+00,
        -1.31544430e+00],
       [-1.26418478e+00,  7.88807586e-01, -1.22655167e+00,
      

In [44]:
# dividir dados entre treino e teste
# treinar no modelo de random forest
# prever e medir acurácia
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(previsores, classe, test_size=0.3, random_state=123)
# depois
# aplicar PCA
# dividir entre treino e teste novamente
# utilizar o random forest novamente
# prever e medir acurária novamente

In [45]:
y_treinamento

array([2, 2, 1, 0, 0, 2, 0, 0, 1, 1, 1, 1, 2, 1, 2, 0, 2, 1, 0, 0, 2, 1,
       2, 2, 0, 1, 1, 2, 0, 2, 1, 1, 0, 2, 2, 0, 0, 1, 1, 2, 0, 0, 1, 0,
       1, 2, 0, 2, 0, 0, 1, 0, 0, 1, 2, 1, 1, 1, 0, 0, 1, 2, 0, 0, 1, 1,
       1, 2, 1, 1, 1, 2, 0, 0, 1, 2, 2, 2, 2, 0, 1, 0, 1, 1, 0, 1, 2, 1,
       2, 2, 0, 1, 0, 2, 2, 1, 1, 2, 2, 1, 0, 1, 1, 2, 2])

In [48]:
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(previsores, classe, test_size=0.3, random_state=1)
y_treinamento

array([2, 0, 0, 0, 1, 0, 0, 2, 2, 2, 2, 2, 1, 2, 1, 0, 2, 2, 0, 0, 2, 0,
       2, 2, 1, 1, 2, 2, 0, 1, 1, 2, 1, 2, 1, 0, 0, 0, 2, 0, 1, 2, 2, 0,
       0, 1, 0, 2, 1, 2, 2, 1, 2, 2, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 2, 2,
       2, 0, 0, 1, 0, 2, 0, 2, 2, 0, 2, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 2, 0, 0, 2, 1, 2, 1, 2, 2, 1, 2, 0])

In [49]:
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(previsores, classe, test_size=0.3, random_state=123)
y_treinamento

array([2, 2, 1, 0, 0, 2, 0, 0, 1, 1, 1, 1, 2, 1, 2, 0, 2, 1, 0, 0, 2, 1,
       2, 2, 0, 1, 1, 2, 0, 2, 1, 1, 0, 2, 2, 0, 0, 1, 1, 2, 0, 0, 1, 0,
       1, 2, 0, 2, 0, 0, 1, 0, 0, 1, 2, 1, 1, 1, 0, 0, 1, 2, 0, 0, 1, 1,
       1, 2, 1, 1, 1, 2, 0, 0, 1, 2, 2, 2, 2, 0, 1, 0, 1, 1, 0, 1, 2, 1,
       2, 2, 0, 1, 0, 2, 2, 1, 1, 2, 2, 1, 0, 1, 1, 2, 2])

In [50]:
# criação do modelo
floresta = RandomForestClassifier(n_estimators=100, random_state = 1234)
floresta.fit(X_treinamento,y_treinamento)
previsoes = floresta.predict(X_teste)
taxa_acerto = accuracy_score(y_teste,previsoes)
taxa_acerto

0.9333333333333333

In [51]:
pca = PCA(n_components=3) # tem que ser menor que o número de atributos original
previsores = pca.fit_transform(previsores) # transformar os previsores originais
previsores # aqui não existe mais significado do ponto de vista de negócio

# criar uma nova modelo de treino e teste para esses novos previsoes
# obs.: utilizaremos o temos test_size e o mesmo random_state
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(previsores, classe, test_size=0.3, random_state=123)
floresta = RandomForestClassifier(n_estimators=100, random_state = 1234)
floresta.fit(X_treinamento,y_treinamento)
previsoes = floresta.predict(X_teste)
taxa_acerto = accuracy_score(y_teste,previsoes)
taxa_acerto

0.9555555555555556